# Fase 2c: verificador v3 (hojas de campo) y diagnóstico de tres clases
Tesis: aplicación móvil para identificar síntomas de ojo de gallo (Rodrigo Alfaro, UCC).

**Por qué existe este cuaderno.** En el teléfono, el verificador v2 rechazó fotos reales de hojas de café tomadas en la
planta (P(hoja) = 0,0009 en una hoja con ojo de gallo), y el diagnóstico binario confundía la roya con ojo de gallo
(31,9 % de las hojas con roya). Este cuaderno:
1. entrena el **verificador v3** agregando hojas de café fotografiadas en campo (RoCoLe) y las hojas con roya;
2. entrena el **diagnóstico v2 de tres clases** (ojo de gallo, roya, sana) con aumento de datos más fuerte;
3. compara las versiones anteriores y nuevas en los mismos conjuntos, incluido PG 26038 (externo, tres clases);
4. convierte ambos modelos a TensorFlow Lite (float16) para la aplicación.

**Cómo usarlo:** Entorno de ejecución → Cambiar tipo de entorno → GPU (T4). Luego Entorno de ejecución → Ejecutar todas.

In [1]:
import os, sys, json, math, time, random, glob, platform, subprocess, shutil
MOCK = os.environ.get("MOCK") == "1"
if not MOCK:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "importlib_resources"], check=False)
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, tensorflow as tf, keras, requests
from PIL import Image, ImageOps, ImageFilter, ImageFile
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import roc_auc_score, confusion_matrix

SEMILLA = 42
TAM = 224
EPOCAS_VER = int(os.environ.get("EPOCAS_VER", 15))
EPOCAS_DIAG = int(os.environ.get("EPOCAS_DIAG", 30))
N_NEG_TFDS = int(os.environ.get("N_NEG", 400))
BASE = os.environ.get("RUTA_BASE", "/content/drive/MyDrive/tesis_ojo_de_gallo")
DS = os.path.join(BASE, "02_Dataset")
TMP = "/content/tmp_fase2c" if not MOCK else "/tmp/tmp_fase2c"
CLASES = ["ojo_de_gallo", "roya", "sana"]            # orden de salida del diagnóstico v2
RUTAS = {
    "ojo_de_gallo": os.path.join(DS, "Ojo_Gallo"),
    "sana": os.path.join(DS, "Sanas"),
    "roya": os.path.join(DS, "Roya"),
    "negativos_propios": os.path.join(DS, "negativos_propios"),
    "bracol": os.path.join(DS, "externos", "BRACOL", "leaf", "images"),
    "pg26038": os.path.join(DS, "externos", "PG26038_muestras", "images"),
    "rocole_drive": os.path.join(DS, "externos"),
    "diag_v1": os.path.join(BASE, "fase2_resultados", "android", "modelo_diagnostico.tflite"),
    "ver_v2": os.path.join(BASE, "fase2b_verificador", "android", "modelo_verificador.tflite"),
}
SAL = os.path.join(BASE, "fase2c_modelos_v3")
for sub in ["tablas", "modelos", "android"]:
    os.makedirs(os.path.join(SAL, sub), exist_ok=True)
os.makedirs(TMP, exist_ok=True)
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def fijar_semillas():
    random.seed(SEMILLA); np.random.seed(SEMILLA); keras.utils.set_random_seed(SEMILLA)
fijar_semillas()
VERSIONES = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
             "numpy": np.__version__, "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}
print(VERSIONES)
for k, r in RUTAS.items():
    print(("OK   " if os.path.exists(r) else "FALTA"), k, r)

def guardar_json(obj, nombre):
    with open(os.path.join(SAL, "tablas", nombre), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=float)

def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

Mounted at /content/drive
{'python': '3.13.16', 'tensorflow': '2.21.0', 'keras': '3.13.2', 'numpy': '2.1.3', 'gpu': ['/physical_device:GPU:0']}
OK    ojo_de_gallo /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Ojo_Gallo
OK    sana /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Sanas
OK    roya /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Roya
OK    negativos_propios /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/negativos_propios
OK    bracol /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos/BRACOL/leaf/images
OK    pg26038 /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos/PG26038_muestras/images
OK    rocole_drive /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos
OK    diag_v1 /content/drive/MyDrive/tesis_ojo_de_gallo/fase2_resultados/android/modelo_diagnostico.tflite
OK    ver_v2 /content/drive/MyDrive/tesis_ojo_de_gallo/fase2b_verificador/android/modelo_verificador.tflite


## 1. Lectura de imágenes (igual que la aplicación)
Orientación EXIF corregida, RGB (PNG transparentes sobre blanco), redimensión bilineal a 224 × 224 con
`tf.image.resize`. Para ahorrar memoria se guardan como enteros de 0 a 255 (diferencia máxima de 0,5 por píxel).

In [2]:
DANADAS = []
def leer(ruta):
    with Image.open(ruta) as im:
        im = ImageOps.exif_transpose(im)
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = im.convert("RGBA")
            fondo = Image.new("RGBA", im.size, (255, 255, 255, 255))
            im = Image.alpha_composite(fondo, im)
        arr = np.asarray(im.convert("RGB"), dtype=np.float32)
    return tf.image.resize(arr[None], (TAM, TAM), method="bilinear").numpy()[0]

def leer_seguro(ruta):
    try:
        return np.clip(np.round(leer(ruta)), 0, 255).astype("uint8")
    except Exception as e:
        DANADAS.append({"imagen": ruta, "error": str(e)}); return None

def leer_varias(rutas, desc=""):
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        res = list(ex.map(leer_seguro, rutas))
    ok = [r for r, x in zip(rutas, res) if x is not None]
    X = [x for x in res if x is not None]
    print(f"{desc}: {len(X)} imágenes leídas de {len(rutas)} en {time.time() - t0:.0f} s")
    return (np.stack(X) if X else np.zeros((0, TAM, TAM, 3), "uint8")), ok

def listar(carpeta, recursivo=False):
    if not os.path.isdir(carpeta):
        return []
    if recursivo:
        return sorted(os.path.join(r, f) for r, _, fs in os.walk(carpeta) for f in fs if f.lower().endswith(EXT))
    return sorted(os.path.join(carpeta, f) for f in os.listdir(carpeta) if f.lower().endswith(EXT))

def ds_desde(X, y, entrenar):
    """Lotes de 32 que se convierten a float32 al vuelo, para no duplicar todo el conjunto en memoria."""
    d = tf.data.Dataset.from_tensor_slices((X, y))
    if entrenar:
        d = d.shuffle(len(X), seed=SEMILLA, reshuffle_each_iteration=True)
    return d.batch(32).map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

def dividir(n, semilla=SEMILLA):
    idx = np.random.RandomState(semilla).permutation(n)
    a = int(round(0.70 * n)); b = a + int(round(0.15 * n))
    return {"train": idx[:a], "validation": idx[a:b], "test": idx[b:]}

## 2. RoCoLe: hojas de café fotografiadas en campo
RoCoLe (Parraga-Alava et al., 2019; Mendeley Data, DOI 10.17632/c5yvn32dzg.2, CC BY 4.0): 1.560 fotos de hojas de café
robusta. Primero se intenta la descarga directa desde Mendeley; si falla, se usa el ZIP que el autor subió a
`02_Dataset/externos/`.

In [3]:
def descomprimir(zp, destino):
    os.makedirs(destino, exist_ok=True)
    r = subprocess.run(["7z", "x", "-y", "-o" + destino, zp], capture_output=True, text=True)
    if r.returncode != 0:
        print("7z:", r.stdout[-500:], r.stderr[-500:])

RO_DIR = os.path.join(TMP, "rocole")
rutas_rocole = listar(RO_DIR, recursivo=True)
if not rutas_rocole and not MOCK:
    zp = os.path.join(TMP, "rocole.zip")
    try:
        with requests.get("https://data.mendeley.com/public-api/zip/c5yvn32dzg/download/2", stream=True, timeout=900) as r:
            print("Mendeley:", r.status_code, r.headers.get("Content-Type"))
            if r.status_code == 200:
                with open(zp, "wb") as f:
                    for ch in r.iter_content(1 << 22):
                        f.write(ch)
        if os.path.exists(zp) and os.path.getsize(zp) > 1e8:
            print("Descargado:", round(os.path.getsize(zp) / 1e9, 2), "GB"); descomprimir(zp, RO_DIR)
    except Exception as e:
        print("Descarga directa falló:", e)
    rutas_rocole = listar(RO_DIR, recursivo=True)
if not rutas_rocole and not MOCK:
    zips = [z for z in glob.glob(os.path.join(RUTAS["rocole_drive"], "*")) if "rocole" in os.path.basename(z).lower() and z.lower().endswith(".zip")]
    print("ZIP de RoCoLe en Drive:", zips)
    for z in zips:
        descomprimir(z, RO_DIR)
    rutas_rocole = listar(RO_DIR, recursivo=True)
if MOCK:
    rutas_rocole = listar(os.path.join(DS, "externos", "RoCoLe"), recursivo=True)
for z in glob.glob(os.path.join(RO_DIR, "**", "*.zip"), recursive=True):   # ZIP dentro del ZIP
    descomprimir(z, os.path.dirname(z)); rutas_rocole = listar(RO_DIR, recursivo=True)
print("Fotos de RoCoLe encontradas:", len(rutas_rocole))
assert len(rutas_rocole) > 0, "No se encontró RoCoLe: sube el ZIP de RoCoLe a 02_Dataset/externos y vuelve a ejecutar esta celda."
print("Ejemplos:", [os.path.relpath(r, RO_DIR) for r in rutas_rocole[:3]])

Mendeley: 200 application/zip
Descargado: 2.25 GB
Fotos de RoCoLe encontradas: 1560
Ejemplos: ['RoCoLe A robusta coffee leaf images dataset/Photos/C10P10E1.jpg', 'RoCoLe A robusta coffee leaf images dataset/Photos/C10P10E2.jpg', 'RoCoLe A robusta coffee leaf images dataset/Photos/C10P10H1.jpg']


## 3. Lectura de todos los conjuntos
- Mendeley: ojo de gallo y sanas con la partición original de la Fase 2; roya con partición 70/15/15 (semilla 42).
- BRACOL y RoCoLe: 70/15/15. PG 26038: solo prueba.

In [4]:
PARTICION = json.loads(r'''{"train": {"ojo_de_gallo": ["20251129_091959.jpg", "20251129_092016.jpg", "20251129_092053.jpg", "20251129_092106.jpg", "20251129_092123.jpg", "20251129_092139.jpg", "20251129_092149.jpg", "20251129_092158.jpg", "20251129_092220.jpg", "20251129_092252.jpg", "20251129_092438.jpg", "20251129_092457.jpg", "20251129_092504.jpg", "20251129_092511.jpg", "20251129_092539.jpg", "20251129_092559.jpg", "20251129_092702.jpg", "20251129_092708.jpg", "20251129_092800.jpg", "20251129_092931.jpg", "20251129_093055.jpg", "20251129_093132.jpg", "20251129_093207.jpg", "20251129_093218.jpg", "20251129_093257.jpg", "20251129_093304.jpg", "20251129_093324.jpg", "20251129_093359.jpg", "20251129_093502.jpg", "20251129_093615.jpg", "20251129_093623.jpg", "20251129_093725.jpg", "20251129_093729.jpg", "20251129_093736.jpg", "20251129_093956.jpg", "20251129_094006.jpg", "20251129_094032.jpg", "20251129_094102.jpg", "20251129_094107.jpg", "20251129_094130.jpg", "20251129_094219.jpg", "20251129_094307.jpg", "20251129_094337.jpg", "20251129_094614.jpg", "20251129_094622.jpg", "20251129_094627.jpg", "20251129_094654.jpg", "20251129_094701.jpg", "20251129_094834.jpg", "20251129_094950.jpg", "20251129_095011.jpg", "20251129_095106.jpg", "20251129_095132.jpg", "20251129_095138.jpg", "20251129_095206.jpg", "20251129_095244.jpg", "20251129_095331.jpg", "20251129_095408.jpg", "20251129_095416.jpg", "20251129_095449.jpg", "20251129_095512.jpg", "20251129_095538.jpg", "20251129_095552.jpg", "20251129_095556.jpg", "20251129_095601.jpg", "20251129_095609.jpg", "20251129_095629.jpg", "20251129_095642.jpg", "20251129_095712.jpg", "20251129_095729.jpg", "20251129_095736.jpg", "20251129_095755.jpg", "20251129_095810.jpg", "20251129_095821.jpg", "20251129_100046.jpg", "20251129_100416.jpg", "20251129_100436.jpg", "20251129_100506.jpg", "20251129_100515.jpg", "20251129_100535.jpg", "20251129_100547.jpg", "20251129_100621.jpg", "20251129_100711.jpg", "20251129_100808.jpg", "20251129_100906.jpg", "20251129_100918.jpg", "20251129_100948.jpg", "20251129_100955.jpg", "20251129_101023.jpg", "20251129_101029.jpg", "20251129_101033.jpg", "20251129_101157.jpg", "20251129_101255.jpg", "20251129_101307.jpg", "20251129_101319.jpg", "20251129_101337.jpg", "20251129_101342.jpg", "20251129_101430.jpg", "20251129_101441.jpg", "20251129_101504.jpg", "20251129_101509.jpg", "20251129_101518.jpg", "20251129_101529.jpg", "20251129_101559.jpg", "20251129_101610.jpg", "20251129_101624.jpg", "20251129_101641.jpg", "20251129_101714.jpg", "20251129_101831.jpg", "20251129_101839.jpg", "20251129_101856.jpg", "20251129_101905.jpg", "20251129_102019.jpg", "20251129_102125.jpg", "20251129_102402.jpg", "20251129_102518.jpg", "20251129_102657.jpg", "20251129_102737.jpg", "20251129_102838.jpg", "20251129_102905.jpg", "20251129_103003.jpg", "20251129_103051.jpg", "20251129_103124.jpg", "20251129_103139.jpg", "20251129_103154.jpg", "20251129_103206.jpg", "20251129_103304.jpg", "20251129_103311.jpg", "20251129_103353.jpg", "20251129_103409.jpg", "20251129_103437.jpg", "20251129_103445.jpg", "20251129_103532.jpg", "20251129_103542.jpg", "20251129_103604.jpg", "20251129_103618.jpg", "20251129_103647.jpg", "20251129_103728.jpg", "20251129_103758.jpg", "20251129_103804.jpg", "20251129_103839.jpg", "20251129_103847.jpg", "20251129_103938.jpg", "20251129_104212.jpg", "20251129_104314.jpg", "20251129_104320.jpg", "20251129_104346.jpg", "20251129_104351.jpg", "20251129_104356.jpg", "20251129_104409.jpg", "20251129_104557.jpg", "20251129_104706.jpg", "20251129_104727.jpg", "20251129_104745.jpg", "20251129_104803.jpg", "20251129_104901.jpg", "20251129_104924.jpg", "20251129_104958.jpg", "20251129_105257.jpg", "20251129_105404.jpg", "20251129_105435.jpg", "20251129_105505.jpg", "20251129_105517.jpg", "20251129_105537.jpg", "20251129_114546.jpg", "20251129_114556.jpg", "20251129_114630.jpg", "20251129_114655.jpg", "20251129_114830.jpg", "20251129_114858.jpg", "20251129_114932.jpg", "20251129_114957.jpg", "20251129_115010.jpg", "20251129_115017.jpg", "20251129_115123.jpg", "20251129_115140.jpg", "20251129_115201.jpg", "20251129_115218.jpg", "20251129_115223.jpg", "20251129_115251.jpg", "20251129_115305.jpg", "20251129_115313.jpg", "20251129_115322.jpg", "20251129_115331.jpg", "20251129_115426.jpg", "20251129_115503.jpg", "20251129_115534.jpg", "20251129_115633.jpg", "20251129_115719.jpg", "20251129_115734.jpg", "20251129_115754.jpg", "20251129_115815.jpg", "20251129_115822.jpg", "20251129_115827.jpg", "20251129_115900.jpg", "20251129_115923.jpg", "20251129_120045.jpg", "20251129_120152.jpg", "20251129_120217.jpg", "20251129_120227.jpg", "20251129_120259.jpg", "20251129_120417.jpg", "20251129_120512.jpg", "20251129_120518.jpg", "20251129_120531.jpg", "20251129_120544.jpg", "20251129_120552.jpg", "20251129_120745.jpg", "20251129_120748.jpg", "20251129_120803.jpg", "20251129_120820.jpg", "20251129_120831.jpg", "20251129_120901.jpg", "20251129_120915.jpg", "20251129_120956.jpg", "20251129_121017.jpg", "20251129_121030.jpg", "20251129_121051.jpg", "20251129_121134.jpg", "20251129_130238.jpg", "20251129_130306.jpg", "20251129_130402.jpg", "20251129_130457.jpg", "20251129_130522.jpg", "20251129_130644.jpg", "20251129_130715.jpg", "20251129_130723.jpg", "20251129_130733.jpg", "20251129_130802.jpg", "20251129_130807.jpg", "20251129_130817.jpg", "20251129_130850.jpg", "20251129_131040.jpg", "20251129_131046.jpg", "20251129_131056.jpg", "20251129_131125.jpg", "20251129_131202.jpg", "20251129_131320.jpg", "20251129_131514.jpg", "20251129_131521.jpg", "20251129_131525.jpg", "20251129_131529.jpg", "20251129_131655.jpg", "20251129_131720.jpg", "20251129_131750.jpg", "20251129_131815.jpg", "20251129_131820.jpg", "20251129_131839.jpg", "20251129_131901.jpg", "20251129_131914.jpg", "20251129_131938.jpg", "20251129_132038.jpg", "20251129_132125.jpg", "20251129_132200.jpg", "20251129_132354.jpg", "20251129_132449.jpg", "20251129_132501.jpg", "20251129_132528.jpg", "20251129_132534.jpg", "20251129_132732.jpg", "20251129_132751.jpg", "20251129_132832.jpg", "20251129_132854.jpg", "20251129_132915.jpg", "20251129_132921.jpg", "20251129_132954.jpg", "20251129_133000.jpg", "20251129_133018.jpg", "20251129_133340.jpg", "20251129_133353.jpg", "20251129_133423.jpg", "20251129_133456.jpg", "20251129_133515.jpg", "20251129_133519.jpg", "20251129_133625.jpg", "20251129_133633.jpg", "20251129_133740.jpg", "20251129_133814.jpg", "20251129_133850.jpg", "20251129_133912.jpg", "20251129_133953.jpg", "20251129_133959.jpg", "20251129_134106.jpg", "20251129_134335.jpg", "20251129_134401.jpg", "20251129_134420.jpg", "20251129_134502.jpg", "20251129_134506.jpg", "20251129_134517.jpg", "20251129_134527.jpg", "20251129_134557.jpg", "20251129_134615.jpg", "20251129_134713.jpg", "20251129_134724.jpg", "20251129_135000.jpg", "20251129_135004.jpg", "20251129_135020.jpg", "20251129_135027.jpg", "20251129_135036.jpg", "20251129_135042.jpg", "20251129_135118.jpg", "20251129_135142.jpg", "20251129_135152.jpg", "20251129_135205.jpg", "20251129_135212.jpg", "20251129_135217.jpg", "20251129_135225.jpg", "20251129_135235.jpg", "20251129_135253.jpg", "20251129_135335.jpg", "20251129_135341.jpg", "20251129_135422.jpg", "20251129_135428.jpg", "20251129_135448.jpg", "20251129_135510.jpg", "20251129_135538.jpg", "20251129_135551.jpg", "20251129_135556.jpg", "20251129_135755.jpg", "20251129_135804.jpg", "20251129_135827.jpg", "20251129_135845.jpg", "20251129_140008.jpg", "20251129_140011.jpg", "20251129_140020.jpg", "20251129_140038.jpg", "20251129_140128.jpg", "20251129_140136.jpg", "20251129_140207.jpg", "20251129_140216.jpg", "20251129_140227.jpg", "20251129_140240.jpg", "20251129_140502.jpg", "20251129_140505.jpg", "20251129_140513.jpg", "20251129_140528.jpg", "20251129_140538.jpg", "20251129_140553.jpg", "20251129_140618.jpg", "20251129_140653.jpg", "20251129_140657.jpg", "20251129_140708.jpg", "20251129_140713.jpg", "20251129_140744.jpg", "20251129_140816.jpg", "20251129_140820.jpg", "20251129_140842.jpg", "20251129_140850.jpg", "20251129_140907.jpg", "20251129_140938.jpg"], "sana": ["Sana 1.jpg", "Sana 100.jpg", "Sana 103.jpg", "Sana 106.jpg", "Sana 107.jpg", "Sana 109.jpg", "Sana 110.jpg", "Sana 111.jpg", "Sana 114.jpg", "Sana 115.jpg", "Sana 118.jpg", "Sana 121.jpg", "Sana 124.jpg", "Sana 125.jpg", "Sana 127.jpg", "Sana 129.jpg", "Sana 131.jpg", "Sana 138.jpg", "Sana 14.jpg", "Sana 141.jpg", "Sana 143.jpg", "Sana 147.jpg", "Sana 148.jpg", "Sana 15.jpg", "Sana 154.jpg", "Sana 156.jpg", "Sana 163.jpg", "Sana 169.jpg", "Sana 173.jpg", "Sana 178.jpg", "Sana 18.jpg", "Sana 183.jpg", "Sana 188.jpg", "Sana 192.jpg", "Sana 194.jpg", "Sana 199.jpg", "Sana 20.jpg", "Sana 201.jpg", "Sana 203.jpg", "Sana 205.jpg", "Sana 206.jpg", "Sana 207.jpg", "Sana 208.jpg", "Sana 214.jpg", "Sana 221.jpg", "Sana 222.jpg", "Sana 223.jpg", "Sana 224.jpg", "Sana 225.jpg", "Sana 227.jpg", "Sana 237.jpg", "Sana 240.jpg", "Sana 241.jpg", "Sana 243.jpg", "Sana 244.jpg", "Sana 245.jpg", "Sana 249.jpg", "Sana 251.jpg", "Sana 253.jpg", "Sana 254.jpg", "Sana 255.jpg", "Sana 258.jpg", "Sana 259.jpg", "Sana 260.jpg", "Sana 261.jpg", "Sana 264.jpg", "Sana 267.jpg", "Sana 268.jpg", "Sana 269.jpg", "Sana 27.jpg", "Sana 272.jpg", "Sana 276.jpg", "Sana 278.jpg", "Sana 288.jpg", "Sana 29.jpg", "Sana 293.jpg", "Sana 294.jpg", "Sana 295.jpg", "Sana 307.jpg", "Sana 311.jpg", "Sana 313.jpg", "Sana 32.jpg", "Sana 323.jpg", "Sana 332.jpg", "Sana 339.jpg", "Sana 34.jpg", "Sana 340.jpg", "Sana 345.jpg", "Sana 346.jpg", "Sana 347.jpg", "Sana 349.jpg", "Sana 35.jpg", "Sana 350.jpg", "Sana 351.jpg", "Sana 36.jpg", "Sana 361.jpg", "Sana 369.jpg", "Sana 37.jpg", "Sana 374.jpg", "Sana 375.jpg", "Sana 377.jpg", "Sana 380.jpg", "Sana 383.jpg", "Sana 384.jpg", "Sana 386.jpg", "Sana 387.jpg", "Sana 388.jpg", "Sana 389.jpg", "Sana 39.jpg", "Sana 392.jpg", "Sana 393.jpg", "Sana 395.jpg", "Sana 399.jpg", "Sana 40.jpg", "Sana 400.jpg", "Sana 401.jpg", "Sana 408.jpg", "Sana 410.jpg", "Sana 411.jpg", "Sana 413.jpg", "Sana 414.jpg", "Sana 419.jpg", "Sana 421.jpg", "Sana 423.jpg", "Sana 425.jpg", "Sana 426.jpg", "Sana 427.jpg", "Sana 43.jpg", "Sana 432.jpg", "Sana 436.jpg", "Sana 438.jpg", "Sana 442.jpg", "Sana 443.jpg", "Sana 446.jpg", "Sana 453.jpg", "Sana 455.jpg", "Sana 457.jpg", "Sana 46.jpg", "Sana 462.jpg", "Sana 464.jpg", "Sana 466.jpg", "Sana 469.jpg", "Sana 473.jpg", "Sana 476.jpg", "Sana 477.jpg", "Sana 48.jpg", "Sana 481.jpg", "Sana 486.jpg", "Sana 488.jpg", "Sana 489.jpg", "Sana 492.jpg", "Sana 496.jpg", "Sana 498.jpg", "Sana 500.jpg", "Sana 507.jpg", "Sana 509.jpg", "Sana 51.jpg", "Sana 511.jpg", "Sana 512.jpg", "Sana 513.jpg", "Sana 514.jpg", "Sana 515.jpg", "Sana 519.jpg", "Sana 52.jpg", "Sana 525.jpg", "Sana 530.jpg", "Sana 531.jpg", "Sana 532.jpg", "Sana 533.jpg", "Sana 536.jpg", "Sana 537.jpg", "Sana 539.jpg", "Sana 548.jpg", "Sana 549.jpg", "Sana 55.jpg", "Sana 551.jpg", "Sana 553.jpg", "Sana 555.jpg", "Sana 556.jpg", "Sana 557.jpg", "Sana 560.jpg", "Sana 565.jpg", "Sana 569.jpg", "Sana 57.jpg", "Sana 572.jpg", "Sana 575.jpg", "Sana 576.jpg", "Sana 577.jpg", "Sana 580.jpg", "Sana 581.jpg", "Sana 582.jpg", "Sana 583.jpg", "Sana 585.jpg", "Sana 586.jpg", "Sana 588.jpg", "Sana 59.jpg", "Sana 591.jpg", "Sana 598.jpg", "Sana 6.jpg", "Sana 60.jpg", "Sana 601.jpg", "Sana 602.jpg", "Sana 608.jpg", "Sana 610.jpg", "Sana 614.jpg", "Sana 617.jpg", "Sana 619.jpg", "Sana 621.jpg", "Sana 622.jpg", "Sana 624.jpg", "Sana 626.jpg", "Sana 629.jpg", "Sana 63.jpg", "Sana 634.jpg", "Sana 637.jpg", "Sana 638.jpg", "Sana 64.jpg", "Sana 643.jpg", "Sana 647.jpg", "Sana 651.jpg", "Sana 653.jpg", "Sana 655.jpg", "Sana 656.jpg", "Sana 657.jpg", "Sana 663.jpg", "Sana 667.jpg", "Sana 668.jpg", "Sana 67.jpg", "Sana 671.jpg", "Sana 672.jpg", "Sana 674.jpg", "Sana 677.jpg", "Sana 682.jpg", "Sana 685.jpg", "Sana 69.jpg", "Sana 691.jpg", "Sana 694.jpg", "Sana 696.jpg", "Sana 7.jpg", "Sana 70.jpg", "Sana 700.jpg", "Sana 701.jpg", "Sana 705.jpg", "Sana 708.jpg", "Sana 715.jpg", "Sana 717.jpg", "Sana 720.jpg", "Sana 723.jpg", "Sana 725.jpg", "Sana 728.jpg", "Sana 73.jpg", "Sana 731.jpg", "Sana 737.jpg", "Sana 739.jpg", "Sana 74.jpg", "Sana 741.jpg", "Sana 743.jpg", "Sana 75.jpg", "Sana 750.jpg", "Sana 752.jpg", "Sana 755.jpg", "Sana 757.jpg", "Sana 762.jpg", "Sana 766.jpg", "Sana 771.jpg", "Sana 775.jpg", "Sana 776.jpg", "Sana 777.jpg", "Sana 779.jpg", "Sana 78.jpg", "Sana 786.jpg", "Sana 788.jpg", "Sana 790.jpg", "Sana 794.jpg", "Sana 795.jpg", "Sana 80.jpg", "Sana 803.jpg", "Sana 808.jpg", "Sana 809.jpg", "Sana 81.jpg", "Sana 813.jpg", "Sana 814.jpg", "Sana 816.jpg", "Sana 817.jpg", "Sana 818.jpg", "Sana 819.jpg", "Sana 820.jpg", "Sana 822.jpg", "Sana 824.jpg", "Sana 827.jpg", "Sana 830.jpg", "Sana 839.jpg", "Sana 84.jpg", "Sana 841.jpg", "Sana 842.jpg", "Sana 843.jpg", "Sana 844.jpg", "Sana 845.jpg", "Sana 846.jpg", "Sana 847.jpg", "Sana 853.jpg", "Sana 857.jpg", "Sana 86.jpg", "Sana 860.jpg", "Sana 861.jpg", "Sana 863.jpg", "Sana 87.jpg", "Sana 873.jpg", "Sana 875.jpg", "Sana 877.jpg", "Sana 886.jpg", "Sana 89.jpg", "Sana 890.jpg", "Sana 892.jpg", "Sana 900.jpg", "Sana 902.jpg", "Sana 903.jpg", "Sana 905.jpg", "Sana 906.jpg", "Sana 91.jpg", "Sana 912.jpg", "Sana 913.jpg", "Sana 917.jpg", "Sana 919.jpg", "Sana 921.jpg", "Sana 928.jpg", "Sana 931.jpg", "Sana 932.jpg", "Sana 934.jpg", "Sana 935.jpg", "Sana 937.jpg", "Sana 938.jpg", "Sana 94.jpg", "Sana 940.jpg", "Sana 943.jpg", "Sana 951.jpg", "Sana 953.jpg", "Sana 956.jpg", "Sana 958.jpg", "Sana 960.jpg", "Sana 966.jpg", "Sana 968.jpg", "Sana 972.jpg", "Sana 976.jpg", "Sana 979.jpg", "Sana 983.jpg", "Sana 99.jpg", "Sana 991.jpg", "Sana 993.jpg"]}, "validation": {"ojo_de_gallo": ["20251129_092009.jpg", "20251129_092333.jpg", "20251129_092735.jpg", "20251129_092925.jpg", "20251129_093509.jpg", "20251129_093848.jpg", "20251129_094057.jpg", "20251129_094345.jpg", "20251129_094716.jpg", "20251129_095022.jpg", "20251129_095346.jpg", "20251129_095547.jpg", "20251129_095626.jpg", "20251129_095703.jpg", "20251129_095949.jpg", "20251129_100037.jpg", "20251129_100540.jpg", "20251129_100648.jpg", "20251129_100822.jpg", "20251129_101002.jpg", "20251129_101038.jpg", "20251129_101044.jpg", "20251129_101243.jpg", "20251129_101733.jpg", "20251129_102732.jpg", "20251129_102831.jpg", "20251129_103110.jpg", "20251129_103945.jpg", "20251129_104403.jpg", "20251129_104738.jpg", "20251129_104907.jpg", "20251129_104918.jpg", "20251129_105430.jpg", "20251129_105445.jpg", "20251129_114552.jpg", "20251129_114708.jpg", "20251129_114714.jpg", "20251129_114939.jpg", "20251129_114946.jpg", "20251129_115257.jpg", "20251129_115342.jpg", "20251129_115513.jpg", "20251129_115618.jpg", "20251129_115807.jpg", "20251129_120332.jpg", "20251129_120401.jpg", "20251129_120633.jpg", "20251129_120920.jpg", "20251129_130229.jpg", "20251129_130232.jpg", "20251129_130322.jpg", "20251129_130533.jpg", "20251129_130657.jpg", "20251129_131129.jpg", "20251129_131356.jpg", "20251129_131402.jpg", "20251129_131638.jpg", "20251129_132216.jpg", "20251129_132305.jpg", "20251129_132632.jpg", "20251129_132910.jpg", "20251129_133030.jpg", "20251129_133527.jpg", "20251129_133840.jpg", "20251129_133933.jpg", "20251129_133943.jpg", "20251129_134656.jpg", "20251129_134708.jpg", "20251129_134753.jpg", "20251129_135137.jpg", "20251129_135531.jpg", "20251129_135606.jpg", "20251129_135624.jpg", "20251129_135927.jpg", "20251129_140749.jpg"], "sana": ["Sana 135.jpg", "Sana 144.jpg", "Sana 155.jpg", "Sana 157.jpg", "Sana 161.jpg", "Sana 19.jpg", "Sana 197.jpg", "Sana 229.jpg", "Sana 239.jpg", "Sana 252.jpg", "Sana 257.jpg", "Sana 297.jpg", "Sana 300.jpg", "Sana 314.jpg", "Sana 316.jpg", "Sana 342.jpg", "Sana 352.jpg", "Sana 355.jpg", "Sana 356.jpg", "Sana 359.jpg", "Sana 379.jpg", "Sana 398.jpg", "Sana 403.jpg", "Sana 428.jpg", "Sana 435.jpg", "Sana 441.jpg", "Sana 467.jpg", "Sana 471.jpg", "Sana 472.jpg", "Sana 50.jpg", "Sana 524.jpg", "Sana 526.jpg", "Sana 528.jpg", "Sana 558.jpg", "Sana 567.jpg", "Sana 574.jpg", "Sana 592.jpg", "Sana 596.jpg", "Sana 611.jpg", "Sana 62.jpg", "Sana 632.jpg", "Sana 633.jpg", "Sana 652.jpg", "Sana 675.jpg", "Sana 684.jpg", "Sana 687.jpg", "Sana 693.jpg", "Sana 697.jpg", "Sana 703.jpg", "Sana 718.jpg", "Sana 72.jpg", "Sana 721.jpg", "Sana 749.jpg", "Sana 765.jpg", "Sana 781.jpg", "Sana 805.jpg", "Sana 826.jpg", "Sana 828.jpg", "Sana 856.jpg", "Sana 866.jpg", "Sana 876.jpg", "Sana 879.jpg", "Sana 882.jpg", "Sana 897.jpg", "Sana 910.jpg", "Sana 936.jpg", "Sana 939.jpg", "Sana 950.jpg", "Sana 957.jpg", "Sana 962.jpg", "Sana 974.jpg", "Sana 980.jpg", "Sana 987.jpg", "Sana 994.jpg"]}, "test": {"ojo_de_gallo": ["20251129_091427.jpg", "20251129_092633.jpg", "20251129_092724.jpg", "20251129_093405.jpg", "20251129_094358.jpg", "20251129_094634.jpg", "20251129_094823.jpg", "20251129_094932.jpg", "20251129_095649.jpg", "20251129_095825.jpg", "20251129_095836.jpg", "20251129_100024.jpg", "20251129_100529.jpg", "20251129_101314.jpg", "20251129_101826.jpg", "20251129_101959.jpg", "20251129_102055.jpg", "20251129_102103.jpg", "20251129_102113.jpg", "20251129_102445.jpg", "20251129_102558.jpg", "20251129_102920.jpg", "20251129_103018.jpg", "20251129_103228.jpg", "20251129_103610.jpg", "20251129_103921.jpg", "20251129_104056.jpg", "20251129_104456.jpg", "20251129_104554.jpg", "20251129_104647.jpg", "20251129_104818.jpg", "20251129_104945.jpg", "20251129_105017.jpg", "20251129_105036.jpg", "20251129_115401.jpg", "20251129_115438.jpg", "20251129_115623.jpg", "20251129_115857.jpg", "20251129_120211.jpg", "20251129_120253.jpg", "20251129_120538.jpg", "20251129_120659.jpg", "20251129_120717.jpg", "20251129_120953.jpg", "20251129_121103.jpg", "20251129_130241.jpg", "20251129_130419.jpg", "20251129_131111.jpg", "20251129_131229.jpg", "20251129_131241.jpg", "20251129_131303.jpg", "20251129_131351.jpg", "20251129_131423.jpg", "20251129_131620.jpg", "20251129_131753.jpg", "20251129_131923.jpg", "20251129_132248.jpg", "20251129_132523.jpg", "20251129_132727.jpg", "20251129_132740.jpg", "20251129_132859.jpg", "20251129_133824.jpg", "20251129_134344.jpg", "20251129_134917.jpg", "20251129_135125.jpg", "20251129_135900.jpg", "20251129_140025.jpg", "20251129_140043.jpg", "20251129_140236.jpg", "20251129_140508.jpg", "20251129_140516.jpg", "20251129_140753.jpg", "20251129_140759.jpg", "20251129_140802.jpg", "20251129_140813.jpg"], "sana": ["Sana 104.jpg", "Sana 119.jpg", "Sana 12.jpg", "Sana 122.jpg", "Sana 123.jpg", "Sana 133.jpg", "Sana 177.jpg", "Sana 187.jpg", "Sana 191.jpg", "Sana 193.jpg", "Sana 198.jpg", "Sana 204.jpg", "Sana 215.jpg", "Sana 23.jpg", "Sana 248.jpg", "Sana 250.jpg", "Sana 275.jpg", "Sana 279.jpg", "Sana 308.jpg", "Sana 309.jpg", "Sana 31.jpg", "Sana 312.jpg", "Sana 320.jpg", "Sana 341.jpg", "Sana 354.jpg", "Sana 360.jpg", "Sana 365.jpg", "Sana 382.jpg", "Sana 417.jpg", "Sana 420.jpg", "Sana 422.jpg", "Sana 431.jpg", "Sana 433.jpg", "Sana 45.jpg", "Sana 452.jpg", "Sana 490.jpg", "Sana 494.jpg", "Sana 516.jpg", "Sana 527.jpg", "Sana 571.jpg", "Sana 594.jpg", "Sana 600.jpg", "Sana 603.jpg", "Sana 615.jpg", "Sana 627.jpg", "Sana 635.jpg", "Sana 636.jpg", "Sana 649.jpg", "Sana 650.jpg", "Sana 660.jpg", "Sana 670.jpg", "Sana 68.jpg", "Sana 689.jpg", "Sana 711.jpg", "Sana 730.jpg", "Sana 732.jpg", "Sana 735.jpg", "Sana 745.jpg", "Sana 761.jpg", "Sana 77.jpg", "Sana 774.jpg", "Sana 79.jpg", "Sana 792.jpg", "Sana 832.jpg", "Sana 837.jpg", "Sana 838.jpg", "Sana 850.jpg", "Sana 869.jpg", "Sana 894.jpg", "Sana 914.jpg", "Sana 933.jpg", "Sana 959.jpg", "Sana 965.jpg", "Sana 97.jpg", "Sana 977.jpg", "Sana 990.jpg"]}}''')
ROYA_TODAS = listar(RUTAS["roya"])
div_r = dividir(len(ROYA_TODAS))
PART = {p: {"ojo_de_gallo": [os.path.join(RUTAS["ojo_de_gallo"], f) for f in PARTICION[p]["ojo_de_gallo"]],
            "sana": [os.path.join(RUTAS["sana"], f) for f in PARTICION[p]["sana"]],
            "roya": [ROYA_TODAS[i] for i in div_r[p]]} for p in ["train", "validation", "test"]}
pd.DataFrame([{"particion": p, "clase": c, "imagen": os.path.basename(r)} for p, d in PART.items() for c, rs in d.items() for r in rs]
             ).to_csv(os.path.join(SAL, "tablas", "particion_tres_clases.csv"), index=False)

MEN = {}
for p in ["train", "validation", "test"]:
    for c in CLASES:
        MEN[(c, p)] = leer_varias(PART[p][c], f"Mendeley {c} {p}")

rutas_bracol = listar(RUTAS["bracol"]); div_b = dividir(len(rutas_bracol))
BRA = {p: leer_varias([rutas_bracol[i] for i in ii], f"BRACOL {p}")[0] for p, ii in div_b.items()}
div_ro = dividir(len(rutas_rocole))
ROC = {p: leer_varias([rutas_rocole[i] for i in ii], f"RoCoLe {p}") for p, ii in div_ro.items()}
pd.DataFrame([{"particion": p, "imagen": os.path.relpath(r, RO_DIR)} for p, (_, rs) in ROC.items() for r in rs]
             ).to_csv(os.path.join(SAL, "tablas", "rocole_particion.csv"), index=False)
PG = {c: leer_varias(listar(os.path.join(RUTAS["pg26038"], c)), f"PG 26038 {c}") for c in ["ojo", "roya", "healthy"]}

Mendeley ojo_de_gallo train: 350 imágenes leídas de 350 en 111 s
Mendeley roya train: 350 imágenes leídas de 350 en 123 s
Mendeley sana train: 349 imágenes leídas de 349 en 93 s
Mendeley ojo_de_gallo validation: 75 imágenes leídas de 75 en 20 s
Mendeley roya validation: 75 imágenes leídas de 75 en 26 s
Mendeley sana validation: 74 imágenes leídas de 74 en 20 s
Mendeley ojo_de_gallo test: 75 imágenes leídas de 75 en 20 s
Mendeley roya test: 74 imágenes leídas de 75 en 21 s
Mendeley sana test: 76 imágenes leídas de 76 en 20 s
BRACOL train: 980 imágenes leídas de 981 en 50 s
BRACOL validation: 210 imágenes leídas de 210 en 6 s
BRACOL test: 211 imágenes leídas de 211 en 7 s
RoCoLe train: 1092 imágenes leídas de 1092 en 83 s
RoCoLe validation: 234 imágenes leídas de 234 en 19 s
RoCoLe test: 234 imágenes leídas de 234 en 20 s
PG 26038 ojo: 40 imágenes leídas de 40 en 5 s
PG 26038 roya: 40 imágenes leídas de 40 en 3 s
PG 26038 healthy: 20 imágenes leídas de 20 en 2 s


## 4. Negativos para el verificador (mismos de la Fase 2b)

In [5]:
def cargar_negativos_tfds(nombre, split, n):
    if MOCK:
        return np.random.RandomState(len(nombre)).randint(0, 255, (n, TAM, TAM, 3)).astype("uint8")
    import tensorflow_datasets as tfds
    ds = tfds.load(nombre, split=split, as_supervised=True, shuffle_files=False).shuffle(1000, seed=SEMILLA).take(n)
    ims = []
    for im, _ in ds:
        x = tf.image.resize(tf.cast(im, tf.float32), (TAM, TAM), method="bilinear").numpy()
        if x.shape[-1] == 1: x = np.repeat(x, 3, -1)
        ims.append(np.clip(np.round(x), 0, 255).astype("uint8"))
    return np.stack(ims)

NEG = {}
for nombre, split, desc in [("rock_paper_scissors", "train", "manos"), ("imagenette/160px", "train", "objetos y escenas"),
                            ("plant_village", "train", "hojas de otros cultivos")]:
    X_n = cargar_negativos_tfds(nombre, split, N_NEG_TFDS)
    idx = np.random.RandomState(SEMILLA).permutation(len(X_n)); a, b = int(0.7 * len(idx)), int(0.85 * len(idx))
    for part, ii in [("train", idx[:a]), ("validation", idx[a:b]), ("test", idx[b:])]:
        NEG[(desc, part)] = X_n[ii]
    print(nombre, len(X_n))

def imagenes_sin_contenido(n, semilla):
    rng = np.random.RandomState(semilla); out = []
    fuente = NEG[("objetos y escenas", "train")]
    for k in range(n):
        t = k % 6
        if t == 0: im = np.ones((TAM, TAM, 3)) * rng.uniform(0, 255, 3)
        elif t == 1:
            g = np.linspace(0, 1, TAM)[:, None, None] if rng.rand() < .5 else np.linspace(0, 1, TAM)[None, :, None]
            c1, c2 = rng.uniform(0, 255, 3), rng.uniform(0, 255, 3); im = np.broadcast_to(c1 + (c2 - c1) * g, (TAM, TAM, 3))
        elif t == 2: im = rng.uniform(0, 255, 3) + rng.normal(0, rng.uniform(3, 40), (TAM, TAM, 3))
        elif t == 3: im = (rng.uniform(0, 25) if rng.rand() < .5 else rng.uniform(230, 255)) + rng.normal(0, 3, (TAM, TAM, 3))
        else: im = np.asarray(Image.fromarray(fuente[rng.randint(len(fuente))]).filter(ImageFilter.GaussianBlur(rng.uniform(12, 30))), "float32")
        out.append(np.clip(np.round(im), 0, 255).astype("uint8"))
    return np.stack(out)

X_sint = imagenes_sin_contenido(300, SEMILLA)
for p, ii in dividir(len(X_sint)).items():
    NEG[("sin contenido", p)] = X_sint[ii]
X_np, rutas_np = leer_varias(listar(RUTAS["negativos_propios"]), "Fotos propias")
idx_np = np.random.RandomState(SEMILLA).permutation(len(X_np)); mitad = len(idx_np) // 2; nv = max(1, mitad // 5)
NEG[("fotos propias", "train")] = X_np[idx_np[nv:mitad]]
NEG[("fotos propias", "validation")] = X_np[idx_np[:nv]]
NEG[("fotos propias", "test")] = X_np[idx_np[mitad:]]
CONTROL_LISAS = np.stack([np.full((TAM, TAM, 3), v, "uint8") for v in (0, 128, 255)])

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/rock_paper_scissors/incomplete.HBRPOI_3.0.0/rock_paper_scissors-train.tfre…

Generating test examples...: 0 examples [00:00, ? examples/s]

... (16 outputs left out)

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.FNO6B9_1.0.2/plant_village-train.tfrecord-[0-9][0…

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.
plant_village 400
Fotos propias: 50 imágenes leídas de 50 en 5 s


## 5. Verificador v3
Hojas de café: Mendeley (tres clases), BRACOL y RoCoLe. No hojas: igual que el v2. Misma receta.

In [6]:
def positivos(p):
    return np.concatenate([MEN[(c, p)][0] for c in CLASES] + [BRA[p], ROC[p][0]])
def negativos(p):
    return np.concatenate([X for (f, q), X in NEG.items() if q == p])
def conjunto_ver(p):
    Xp, Xn = positivos(p), negativos(p)
    return np.concatenate([Xp, Xn]), np.concatenate([np.ones(len(Xp)), np.zeros(len(Xn))]).astype("float32")

Xtr, ytr = conjunto_ver("train"); Xva, yva = conjunto_ver("validation")
print("Verificador: entrenamiento", Xtr.shape, "hojas", int(ytr.sum()), "| validación", Xva.shape)
pesos_v = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * (ytr == 1).sum())}

def base_mobilenet():
    b = keras.applications.MobileNetV3Small(input_shape=(TAM, TAM, 3), include_top=False,
                                            weights=None if MOCK else "imagenet", include_preprocessing=True)
    b.trainable = False
    return b

fijar_semillas()
e = keras.Input((TAM, TAM, 3), name="imagen")
aum_v = keras.Sequential([keras.layers.RandomFlip("horizontal"), keras.layers.RandomRotation(0.1),
                          keras.layers.RandomZoom(0.1)], name="data_augmentation")
z = base_mobilenet()(aum_v(e), training=False)
z = keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(z)
z = keras.layers.Dropout(0.2, name="dropout")(z)
z = keras.layers.Dense(1, activation="sigmoid", name="es_hoja_de_cafe")(z)
ver3 = keras.Model(e, z, name="verificador_v3")
ver3.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy", keras.metrics.AUC(name="auc")])
h = ver3.fit(ds_desde(Xtr, ytr, True), validation_data=ds_desde(Xva, yva, False), epochs=EPOCAS_VER,
             class_weight=pesos_v, verbose=2)
guardar_json({k: [float(v) for v in vs] for k, vs in h.history.items()}, "verificador_v3_historial.json")
del Xtr, Xva; import gc; gc.collect()

Verificador: entrenamiento (4191, 224, 224, 3) hojas 3121 | validación (898, 224, 224, 3)
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
131/131 - 28s - 217ms/step - accuracy: 0.8442 - auc: 0.9034 - loss: 0.4108 - val_accuracy: 0.9744 - val_auc: 0.9983 - val_loss: 0.1644
Epoch 2/15
131/131 - 7s - 52ms/step - accuracy: 0.9814 - auc: 0.9984 - loss: 0.1479 - val_accuracy: 0.9944 - val_auc: 0.9998 - val_loss: 0.0965
Epoch 3/15
131/131 - 5s - 39ms/step - accuracy: 0.9926 - auc: 0.9993 - loss: 0.0889 - val_accuracy: 0.9967 - val_auc: 0.9999 - val_loss: 0.0655
Epoch 4/15
131/131 - 4s - 31ms/step - accuracy: 0.9952 - auc: 0.9999 - loss: 0.0596 - val_accuracy: 0.9989 - val_auc: 1.0000 - val_loss: 0.0410
Epoch 5/15
131/131 - 4s - 32ms/step - accuracy: 0.9969 - auc: 0.9999 - loss: 0.0476 - val_accuracy: 0.9989 - val_auc: 1.0000 - val_loss: 0.0372
Epoch 6/15
131/131 - 4s - 32ms/step - accuracy: 0.9971 - auc: 0.9999 - loss: 0.0378 - val_accuracy: 0.9989 - val_auc: 1.0000 - val_loss: 0.

971

## 6. Diagnóstico v2 de tres clases
MobileNetV3Small con la base congelada, salida softmax de tres clases (ojo de gallo, roya, sana) y aumento de datos
más fuerte: giros completos, recortes, traslación, brillo y contraste. Adam 1e-3, 30 épocas, semilla 42; se conserva
el modelo de la época con menor pérdida de validación.

In [7]:
def conjunto_diag(p):
    X = np.concatenate([MEN[(c, p)][0] for c in CLASES])
    y = np.concatenate([np.full(len(MEN[(c, p)][0]), i) for i, c in enumerate(CLASES)])
    return X, y
Xd_tr, yd_tr = conjunto_diag("train"); Xd_va, yd_va = conjunto_diag("validation"); Xd_te, yd_te = conjunto_diag("test")
print("Diagnóstico:", {p: len(conjunto_diag(p)[1]) for p in ["train", "validation", "test"]})

capas_aum = [keras.layers.RandomFlip("horizontal_and_vertical"), keras.layers.RandomRotation(0.5),
             keras.layers.RandomZoom((-0.3, 0.1)), keras.layers.RandomTranslation(0.1, 0.1),
             keras.layers.RandomContrast(0.3), keras.layers.RandomBrightness(0.3, value_range=(0, 255))]
fijar_semillas()
e = keras.Input((TAM, TAM, 3), name="imagen")
z = base_mobilenet()(keras.Sequential(capas_aum, name="data_augmentation")(e), training=False)
z = keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(z)
z = keras.layers.Dropout(0.2, name="dropout")(z)
z = keras.layers.Dense(len(CLASES), activation="softmax", name="clasificacion")(z)
diag2 = keras.Model(e, z, name="diagnostico_tres_clases")
diag2.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mejor = os.path.join(TMP, "diag2_mejor.weights.h5")
h = diag2.fit(ds_desde(Xd_tr, yd_tr, True), validation_data=ds_desde(Xd_va, yd_va, False), epochs=EPOCAS_DIAG,
              verbose=2,
              callbacks=[keras.callbacks.ModelCheckpoint(mejor, monitor="val_loss", save_best_only=True, save_weights_only=True)])
diag2.load_weights(mejor)
guardar_json({k: [float(v) for v in vs] for k, vs in h.history.items()}, "diagnostico_v2_historial.json")

Diagnóstico: {'train': 1049, 'validation': 224, 'test': 225}
Epoch 1/30
33/33 - 9s - 260ms/step - accuracy: 0.5529 - loss: 0.9448 - val_accuracy: 0.7455 - val_loss: 0.5929
Epoch 2/30
33/33 - 2s - 48ms/step - accuracy: 0.7607 - loss: 0.5875 - val_accuracy: 0.7857 - val_loss: 0.4493
Epoch 3/30
33/33 - 2s - 47ms/step - accuracy: 0.8179 - loss: 0.4711 - val_accuracy: 0.8170 - val_loss: 0.4003
Epoch 4/30
33/33 - 2s - 47ms/step - accuracy: 0.8561 - loss: 0.3959 - val_accuracy: 0.8393 - val_loss: 0.3627
Epoch 5/30
33/33 - 2s - 48ms/step - accuracy: 0.8742 - loss: 0.3551 - val_accuracy: 0.8527 - val_loss: 0.3157
Epoch 6/30
33/33 - 1s - 35ms/step - accuracy: 0.8808 - loss: 0.3196 - val_accuracy: 0.8482 - val_loss: 0.3318
Epoch 7/30
33/33 - 1s - 45ms/step - accuracy: 0.8942 - loss: 0.3008 - val_accuracy: 0.8571 - val_loss: 0.2968
Epoch 8/30
33/33 - 2s - 67ms/step - accuracy: 0.9133 - loss: 0.2599 - val_accuracy: 0.8482 - val_loss: 0.3048
Epoch 9/30
33/33 - 2s - 47ms/step - accuracy: 0.9142 - los

## 7. Conversión a TensorFlow Lite (float16)

In [8]:
def inferencia(m):
    ent = keras.Input((TAM, TAM, 3), name="imagen"); x = ent
    for capa in m.layers[1:]:
        if capa.name == "data_augmentation":
            continue
        x = capa(x, training=False)
    return keras.Model(ent, x)

def a_tflite(m, nombre):
    carpeta = os.path.join(TMP, "sm_" + nombre)
    try: m.export(carpeta, verbose=False)
    except TypeError: m.export(carpeta)
    c = tf.lite.TFLiteConverter.from_saved_model(carpeta)
    c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]
    return c.convert()

class TFL:
    def __init__(self, contenido):
        self.it = tf.lite.Interpreter(model_content=contenido); self.it.allocate_tensors()
        self.i = self.it.get_input_details()[0]; self.o = self.it.get_output_details()[0]
    def __call__(self, X):
        out = []
        for k in range(len(X)):
            self.it.set_tensor(self.i["index"], X[k:k + 1].astype("float32")); self.it.invoke()
            out.append(self.it.get_tensor(self.o["index"])[0].copy())
        return np.array(out)

ver3_inf, diag2_inf = inferencia(ver3), inferencia(diag2)
B_VER3, B_DIAG2 = a_tflite(ver3_inf, "ver3"), a_tflite(diag2_inf, "diag2")
VER = {"v3": TFL(B_VER3)}; DIAG = {"v2_tres_clases": TFL(B_DIAG2)}
if os.path.isfile(RUTAS["ver_v2"]): VER["v2"] = TFL(open(RUTAS["ver_v2"], "rb").read())
if os.path.isfile(RUTAS["diag_v1"]): DIAG["v1_binario"] = TFL(open(RUTAS["diag_v1"], "rb").read())
# Acuerdo TFLite vs Keras en la prueba de Mendeley
ref = diag2_inf.predict(ds_desde(Xd_te, yd_te, False), verbose=0); tfl = DIAG["v2_tres_clases"](Xd_te)
ref_v = ver3_inf.predict(positivos("test").astype("float32"), verbose=0).ravel(); tfl_v = VER["v3"](positivos("test")).ravel()
conv = {"diag_acuerdo_clase": float((ref.argmax(1) == tfl.argmax(1)).mean()), "diag_dif_max": float(np.abs(ref - tfl).max()),
        "ver_acuerdo": float(((ref_v >= .5) == (tfl_v >= .5)).mean()), "ver_dif_max": float(np.abs(ref_v - tfl_v).max()),
        "MB_verificador": len(B_VER3) / 1e6, "MB_diagnostico": len(B_DIAG2) / 1e6}
print(conv); guardar_json(conv, "conversion_tflite.json")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


{'diag_acuerdo_clase': 1.0, 'diag_dif_max': 0.014568507671356201, 'ver_acuerdo': 1.0, 'ver_dif_max': 0.008671879768371582, 'MB_verificador': 1.943624, 'MB_diagnostico': 1.94602}


## 8. Comparación de verificadores (v2 frente a v3)

In [9]:
PRUEBAS_VER = [("Hojas Mendeley (prueba, 3 clases)", np.concatenate([MEN[(c, "test")][0] for c in CLASES]), 1),
               ("Hojas BRACOL (prueba)", BRA["test"], 1),
               ("Hojas RoCoLe en campo (prueba)", ROC["test"][0], 1),
               ("Hojas PG 26038 (externas)", np.concatenate([PG[c][0] for c in PG]), 1),
               ("Manos (prueba)", NEG[("manos", "test")], 0),
               ("Objetos y escenas (prueba)", NEG[("objetos y escenas", "test")], 0),
               ("Hojas de otros cultivos (prueba)", NEG[("hojas de otros cultivos", "test")], 0),
               ("Imágenes sin contenido (prueba)", NEG[("sin contenido", "test")], 0),
               ("Fotos propias (prueba)", NEG[("fotos propias", "test")], 0),
               ("Control gris, negra y blanca", CONTROL_LISAS, 0)]
filas = []
for nv, m in VER.items():
    for nombre, X, clase in PRUEBAS_VER:
        p = m(X).ravel(); ok = (p >= .5) if clase == 1 else (p < .5); k = int(ok.sum()); lo, hi = wilson(k, len(p))
        filas.append({"verificador": nv, "conjunto": nombre, "n": len(p), "medida": "aceptadas" if clase else "rechazadas",
                      "aciertos": k, "tasa": k / len(p), "IC95_inf": lo, "IC95_sup": hi})
df_v = pd.DataFrame(filas); df_v.to_csv(os.path.join(SAL, "tablas", "comparacion_verificador_v2_v3.csv"), index=False)
print(df_v.pivot_table(index="conjunto", columns="verificador", values="tasa", sort=False).round(3).to_string())

verificador                          v3     v2
conjunto                                      
Hojas Mendeley (prueba, 3 clases)  1.00  0.991
Hojas BRACOL (prueba)              1.00  1.000
Hojas RoCoLe en campo (prueba)     1.00  0.282
Hojas PG 26038 (externas)          0.89  0.910
Manos (prueba)                     1.00  1.000
Objetos y escenas (prueba)         1.00  1.000
Hojas de otros cultivos (prueba)   1.00  1.000
Imágenes sin contenido (prueba)    1.00  1.000
Fotos propias (prueba)             0.92  1.000
Control gris, negra y blanca       1.00  1.000


## 9. Comparación de diagnósticos (binario v1 frente a tres clases v2)
Regla de la aplicación. Binario: P(ojo) ≥ 0,70 compatible; ≤ 0,30 sin síntomas; en medio, no concluyente.
Tres clases: la clase con probabilidad ≥ 0,70 decide (ojo de gallo, roya o sana); si ninguna llega, no concluyente.
Métricas de ojo de gallo con umbral 0,5 sobre P(ojo de gallo): sensibilidad (ojo de gallo detectado) y especificidad
(sanas y roya no marcadas como ojo de gallo).

In [10]:
def regla(nombre, s):
    if nombre == "v1_binario":
        po = 1 - s.ravel()
        return po, np.where(po >= .7, "ojo_de_gallo", np.where(po <= .3, "sana", "no_concluyente"))
    po = s[:, 0]; mx = s.max(1)
    return po, np.where(mx >= .7, np.array(CLASES)[s.argmax(1)], "no_concluyente")

EVAL = {"Mendeley prueba": {c: MEN[(c, "test")][0] for c in CLASES},
        "PG 26038 externo": {"ojo_de_gallo": PG["ojo"][0], "roya": PG["roya"][0], "sana": PG["healthy"][0]}}
filas, conteos = [], []
for nd, m in DIAG.items():
    for ne, d in EVAL.items():
        po_all, y_all = [], []
        for c, X in d.items():
            po, r = regla(nd, m(X))
            po_all.append(po); y_all.append(np.full(len(po), c))
            for etiqueta, n in pd.Series(r).value_counts().items():
                conteos.append({"diagnostico": nd, "conjunto": ne, "clase_real": c, "resultado_app": etiqueta, "n": int(n)})
        po_all = np.concatenate(po_all); y_all = np.concatenate(y_all); es_ojo = y_all == "ojo_de_gallo"
        vp = int((po_all[es_ojo] >= .5).sum()); vn = int((po_all[~es_ojo] < .5).sum())
        roya = y_all == "roya"; roya_como_ojo = int((po_all[roya] >= .5).sum())
        filas.append({"diagnostico": nd, "conjunto": ne, "n_ojo": int(es_ojo.sum()), "n_no_ojo": int((~es_ojo).sum()),
                      "sensibilidad": vp / es_ojo.sum(), "sens_IC95": wilson(vp, int(es_ojo.sum())),
                      "especificidad": vn / (~es_ojo).sum(), "esp_IC95": wilson(vn, int((~es_ojo).sum())),
                      "auc_ojo": roc_auc_score(es_ojo, po_all), "roya_como_ojo": roya_como_ojo / max(1, roya.sum())})
df_d = pd.DataFrame(filas); df_d.to_csv(os.path.join(SAL, "tablas", "comparacion_diagnostico_v1_v2.csv"), index=False)
df_c = pd.DataFrame(conteos); df_c.to_csv(os.path.join(SAL, "tablas", "regla_app_por_clase.csv"), index=False)
print(df_d.round(3).to_string(index=False))
print(df_c.pivot_table(index=["diagnostico", "conjunto", "clase_real"], columns="resultado_app", values="n", fill_value=0).to_string())

   diagnostico         conjunto  n_ojo  n_no_ojo  sensibilidad                                   sens_IC95  especificidad                                 esp_IC95  auc_ojo  roya_como_ojo
v2_tres_clases  Mendeley prueba     75       150         1.000                   (0.9512761575171942, 1.0)          1.000                (0.9750297556319235, 1.0)    1.000          0.000
v2_tres_clases PG 26038 externo     40        60         0.075 (0.025836025774588198, 0.19864233524310543)          1.000   (0.93982814785791, 0.9999999999999998)    0.812          0.000
    v1_binario  Mendeley prueba     75       150         1.000                   (0.9512761575171942, 1.0)          0.773 (0.6999980606806022, 0.8330182057315159)    0.974          0.446
    v1_binario PG 26038 externo     40        60         0.650   (0.49505880837257693, 0.7786547112682372)          0.850 (0.7388541093022145, 0.9190255941983225)    0.816          0.200
resultado_app                                 no_concluyente  ojo

## 10. Filtro de calidad y archivos para la aplicación

In [11]:
def desviaciones(X, lote=256):
    return np.concatenate([X[i:i + lote].reshape(len(X[i:i + lote]), -1).astype("float32").std(1) for i in range(0, len(X), lote)])
d = np.concatenate([desviaciones(positivos(p)) for p in ["train", "validation", "test"]] + [desviaciones(PG[c][0]) for c in PG])
UMBRAL_DESVIACION = float(np.floor(d.min() / 2 * 10) / 10)
filtro = {"n_hojas": len(d), "desviacion_min": float(d.min()), "desviacion_p1": float(np.percentile(d, 1)), "umbral": UMBRAL_DESVIACION}
print(filtro); guardar_json(filtro, "filtro_calidad.json")
open(os.path.join(SAL, "android", "modelo_verificador.tflite"), "wb").write(B_VER3)
open(os.path.join(SAL, "android", "modelo_diagnostico.tflite"), "wb").write(B_DIAG2)
contrato = {"verificador": "salida 1: P(hoja de café)", "diagnostico": {"salidas": CLASES, "tipo": "softmax"},
            "entrada": "224x224 RGB float32 0-255, EXIF corregido, bilineal", "umbral_desviacion": UMBRAL_DESVIACION}
guardar_json(contrato, "contrato_app.json")
guardar_json({"versiones": VERSIONES, "danadas": DANADAS, "conteo_rocole": len(rutas_rocole)}, "resumen_fase2c.json")
print("Modelos para la app en", os.path.join(SAL, "android"))

{'n_hojas': 4559, 'desviacion_min': 27.052473068237305, 'desviacion_p1': 40.08843994140625, 'umbral': 13.5}
Modelos para la app en /content/drive/MyDrive/tesis_ojo_de_gallo/fase2c_modelos_v3/android
